In [1]:
import pandas as pd
cpcb_df = pd.read_csv(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\raw\cpcb\merged.csv")
cpcb_df['date'] = pd.to_datetime(cpcb_df['date']).dt.date

stations = cpcb_df[['station_name', 'latitude', 'longitude']].drop_duplicates()
print(f"Total CPCB stations : {len(stations)}")
print(f"Total CPCB rows     : {len(cpcb_df)}")
print(f"Date range          : {cpcb_df['date'].min()} to {cpcb_df['date'].max()}")
print(f"PM25 missing %      : {cpcb_df['pm25'].isna().mean()*100:.1f}%")

# This tells you the realistic ceiling for your final merged dataset
print(f"\nExpected merged rows: ~{len(cpcb_df):,}")

Total CPCB stations : 98
Total CPCB rows     : 67970
Date range          : 2023-01-01 to 2024-12-31
PM25 missing %      : 0.0%

Expected merged rows: ~67,970


In [2]:
import pandas as pd

FILE = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_dataset.parquet"

df = pd.read_parquet(FILE)

print(df.head())

         date     station_name        pm25  latitude longitude  n_pixels  \
0  2023-01-01  Sector 62 Noida  131.027604    28.628   77.3649       5.0   
1  2023-01-02  Sector 62 Noida  204.667292    28.628   77.3649       0.0   
2  2023-01-03  Sector 62 Noida  232.663958    28.628   77.3649       5.0   
3  2023-01-04  Sector 62 Noida  182.754687    28.628   77.3649       6.0   
4  2023-01-05  Sector 62 Noida  259.587500    28.628   77.3649       5.0   

   AOD_mean   AOD_max   AOD_p75  AOD_count  ...  SPEED_mean  SPEED_min  \
0  0.978464  1.244282  0.978464        7.0  ...    1.996576   1.197900   
1       NaN       NaN       NaN        0.0  ...    2.732568   1.221766   
2  2.416098  2.570632  2.416098        3.0  ...    5.108914   2.726804   
3  1.111868  1.203726  1.111868        4.0  ...    4.043421   2.261521   
4  1.452860  1.886003  1.452860        4.0  ...    3.574554   2.451186   

    PRECTOT_sum  vent_coeff_mean  vent_coeff_min  inversion_proxy_mean  \
0  1.446804e-17       74

In [4]:
df = pd.read_parquet(r'C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_dataset.parquet')
df['date'] = pd.to_datetime(df['date'])

# See exactly which columns have missing values
print("=== All missing values ===")
miss = df.isna().mean() * 100
print(miss[miss > 0].sort_values(ascending=False))

# Check your column names — critical before feature engineering
print("\n=== All columns ===")
print(df.columns.tolist())

print("\n=== Dtypes ===")
print(df.dtypes)

print("\n=== PM25 stats ===")
print(df['pm25'].describe().round(1))

=== All missing values ===
AOD_mean                44.020892
AOD_max                 44.020892
AOD_p75                 44.020892
n_pixels                 4.403413
AOD_count                4.403413
AOD_coverage             4.403413
PBLH_mean                0.135354
PBLH_min                 0.135354
PBLH_max                 0.135354
TLML_mean                0.135354
TLML_max                 0.135354
moisture_mean            0.135354
moisture_max             0.135354
SPEED_mean               0.135354
SPEED_min                0.135354
PRECTOT_sum              0.135354
vent_coeff_mean          0.135354
vent_coeff_min           0.135354
inversion_proxy_mean     0.135354
inversion_proxy_max      0.135354
wind_dir_mean            0.135354
PBLH_morning_mean        0.135354
PBLH_morning_min         0.135354
dtype: float64

=== All columns ===
['date', 'station_name', 'pm25', 'latitude', 'longitude', 'n_pixels', 'AOD_mean', 'AOD_max', 'AOD_p75', 'AOD_count', 'AOD_coverage', 'PBLH_mean', 'PBLH_min

In [5]:
# 0.13% = ~91 rows — fix with station-month median
merra_cols = [
    'PBLH_mean', 'PBLH_min', 'PBLH_max',
    'TLML_mean', 'TLML_max',
    'moisture_mean', 'moisture_max',
    'SPEED_mean', 'SPEED_min',
    'PRECTOT_sum',
    'vent_coeff_mean', 'vent_coeff_min',
    'inversion_proxy_mean', 'inversion_proxy_max',
    'wind_dir_mean',
    'PBLH_morning_mean', 'PBLH_morning_min'
]

# Keep only cols that actually exist in your df
merra_cols = [c for c in merra_cols if c in df.columns]

df['month'] = df['date'].dt.month  # need this for groupby

for col in merra_cols:
    if df[col].isna().any():
        # Level 1: station + month median
        df[col] = df.groupby(['station_name', 'month'])[col].transform(
            lambda x: x.fillna(x.median())
        )
        # Level 2: global month median (fallback)
        df[col] = df.groupby('month')[col].transform(
            lambda x: x.fillna(x.median())
        )
        # Level 3: global median (last resort)
        df[col] = df[col].fillna(df[col].median())

print(f"MERRA missing after fix: {df[merra_cols].isna().sum().sum()}")  # must be 0

MERRA missing after fix: 0


In [6]:
# 44% missing AOD = cloud/night days — DO NOT drop, fill with -1 sentinel
aod_value_cols = ['AOD_mean', 'AOD_max', 'AOD_p75']
# Only fill cols that exist
aod_value_cols = [c for c in aod_value_cols if c in df.columns]

df[aod_value_cols] = df[aod_value_cols].fillna(-1)

print(f"AOD_mean = -1 (cloud) : {(df['AOD_mean']==-1).mean()*100:.1f}%")
print(f"AOD_mean > 0 (clear)  : {(df['AOD_mean'] > 0).mean()*100:.1f}%")
print(f"AOD NaN remaining     : {df['AOD_mean'].isna().sum()}")  # must be 0

AOD_mean = -1 (cloud) : 44.0%
AOD_mean > 0 (clear)  : 56.0%
AOD NaN remaining     : 0


In [7]:
remaining = df.isna().sum()
remaining = remaining[remaining > 0]

if len(remaining) == 0:
    print("Zero missing values ✓  — ready for feature engineering")
else:
    print("Still missing — fix before proceeding:")
    print(remaining)

Still missing — fix before proceeding:
n_pixels        2993
AOD_count       2993
AOD_coverage    2993
dtype: int64


In [9]:
# Feature engineering — create new features based on domain knowledge and interactions
import numpy as np

# ── Temporal ──────────────────────────────────────────────────────────────
df['day_of_year'] = df['date'].dt.dayofyear
df['day_of_week'] = df['date'].dt.dayofweek
df['year']        = df['date'].dt.year
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

season_map = {12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}
df['season'] = df['month'].map(season_map)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['doy_sin']   = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['doy_cos']   = np.cos(2 * np.pi * df['day_of_year'] / 365)

# ── Safe interaction — only fires on real AOD, not sentinel ───────────────
def safe_interact(aod_col, met_col, op='divide'):
    valid  = df[aod_col] != -1
    result = pd.Series(-1.0, index=df.index)
    if op == 'divide':
        result[valid] = df.loc[valid, aod_col] / (df.loc[valid, met_col] + 1)
    elif op == 'multiply':
        result[valid] = df.loc[valid, aod_col] * df.loc[valid, met_col]
    return result

# ── Physics interactions ───────────────────────────────────────────────────
df['PM_proxy']        = safe_interact('AOD_mean', 'PBLH_morning_min', 'divide')
df['PM_proxy_max']    = safe_interact('AOD_max',  'PBLH_morning_min', 'divide')
df['AOD_x_moisture']  = safe_interact('AOD_mean', 'moisture_mean',    'multiply')
df['AOD_x_vent']      = safe_interact('AOD_mean', 'vent_coeff_mean',  'divide')
df['AOD_x_vent_min']  = safe_interact('AOD_mean', 'vent_coeff_min',   'divide')
df['AOD_x_inversion'] = safe_interact('AOD_mean', 'inversion_proxy_max', 'multiply')
df['AOD_x_PBLH_min']  = safe_interact('AOD_mean', 'PBLH_min',         'divide')

# ── Pollution trap score ───────────────────────────────────────────────────
df['trap_score'] = (
    df['inversion_proxy_max'] /
    df['SPEED_min'].clip(lower=0.1) /
    (df['PBLH_morning_min'] + 1)
)

df['is_stagnant'] = (
    (df['SPEED_mean'] < 2.0) &
    (df['PBLH_min']   < 500)
).astype(int)

# ── Precipitation washout ──────────────────────────────────────────────────
df = df.sort_values(['station_name', 'date']).reset_index(drop=True)

df['rain_lag1']  = df.groupby('station_name')['PRECTOT_sum'].shift(1).fillna(0)
df['rain_lag2']  = df.groupby('station_name')['PRECTOT_sum'].shift(2).fillna(0)
df['is_rainy']   = (df['PRECTOT_sum'] > 0.5).astype(int)
df['post_rain1'] = (df['rain_lag1'] > 0.5).astype(int)
df['post_rain2'] = (df['rain_lag2'] > 0.5).astype(int)
df['rain_3day']  = df['PRECTOT_sum'].fillna(0) + df['rain_lag1'] + df['rain_lag2']

# ── AOD lag and rolling ────────────────────────────────────────────────────
aod_real = df['AOD_mean'].replace(-1, np.nan)  # temp restore NaN for rolling

df['AOD_lag1'] = df.groupby('station_name')['AOD_mean'].shift(1).fillna(-1)
df['AOD_lag2'] = df.groupby('station_name')['AOD_mean'].shift(2).fillna(-1)

df['AOD_roll3'] = (
    aod_real.groupby(df['station_name'])
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
    .fillna(-1)
)
df['AOD_roll7'] = (
    aod_real.groupby(df['station_name'])
    .transform(lambda x: x.rolling(7, min_periods=3).mean())
    .fillna(-1)
)

# ── Station spatial context ────────────────────────────────────────────────
df['station_pm_mean']  = df.groupby('station_name')['pm25'].transform('mean')
df['station_pm_std']   = df.groupby('station_name')['pm25'].transform('std')
df['station_month_pm'] = df.groupby(
    ['station_name', 'month']
)['pm25'].transform('mean')

df['proxy_anomaly'] = df['PM_proxy'] - df.groupby(
    ['station_name', 'month']
)['PM_proxy'].transform('mean')

# Geography — IGP vs coastal vs Deccan gradient
df['lat'] = df['latitude']
df['lon']  = df['longitude']

print(f"After feature engineering: {df.shape}")
print(f"New features added: {df.shape[1] - 28}")

After feature engineering: (67970, 63)
New features added: 35


In [11]:
# n_pixels = 0 means no satellite coverage that day
# AOD_count = 0 means no valid retrievals
# AOD_coverage = 0 means 0% sky coverage

df['n_pixels']    = df['n_pixels'].fillna(0).astype(int)
df['AOD_count']   = df['AOD_count'].fillna(0).astype(int)
df['AOD_coverage']= df['AOD_coverage'].fillna(0.0)

# Verify
remaining = df.isna().sum()
remaining = remaining[remaining > 0]

if len(remaining) == 0:
    print("Zero missing values ✓ — ready for feature engineering")
else:
    print("Still missing:")
    print(remaining)

Zero missing values ✓ — ready for feature engineering


In [12]:
# Drop rows where PM25 missing
before = len(df)
df = df.dropna(subset=['pm25'])
print(f"Dropped {before - len(df)} rows (missing PM25)")

# Remove PM25 outliers
p995 = df['pm25'].quantile(0.995)
p005 = df['pm25'].quantile(0.005)
df   = df[(df['pm25'] >= p005) & (df['pm25'] <= p995)]
print(f"PM25 range: {df['pm25'].min():.1f} – {df['pm25'].max():.1f} µg/m³")

# Final zero-missing check
assert df.isna().sum().sum() == 0, f"Still have missing: {df.isna().sum()[df.isna().sum()>0]}"
print(f"\nFinal shape : {df.shape}")
print(f"Zero missing ✓")

df.to_parquet(
    r'C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet',
    index=False
)

Dropped 0 rows (missing PM25)
PM25 range: 4.9 – 268.1 µg/m³

Final shape : (66617, 63)
Zero missing ✓


In [13]:
df3 = pd.read_parquet(
    r'C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet'
)

In [14]:
df3.head()

,date,station_name,pm25,latitude,longitude,n_pixels,AOD_mean,AOD_max,AOD_p75,AOD_count,...,AOD_lag1,AOD_lag2,AOD_roll3,AOD_roll7,station_pm_mean,station_pm_std,station_month_pm,proxy_anomaly,lat,lon
0,2023-01-03,AIIMS Raipur,54.811250,21.2575,81.5775,7,2.062045,2.549567,2.062045,7,...,-1.000000,-1.000000,2.062045,-1.000000,28.675352,18.036318,47.799848,0.079515,21.2575,81.5775
1,2023-01-04,AIIMS Raipur,65.033958,21.2575,81.5775,0,-1.000000,-1.000000,-1.000000,0,...,2.062045,-1.000000,2.062045,-1.000000,28.675352,18.036318,47.799848,-0.940005,21.2575,81.5775
2,2023-01-05,AIIMS Raipur,63.587917,21.2575,81.5775,7,2.145734,2.696037,2.145734,6,...,-1.000000,2.062045,2.103889,-1.000000,28.675352,18.036318,47.799848,0.071047,21.2575,81.5775
3,2023-01-06,AIIMS Raipur,56.248542,21.2575,81.5775,7,0.639584,0.949890,0.639584,7,...,2.145734,-1.000000,1.392659,1.615788,28.675352,18.036318,47.799848,0.070095,21.2575,81.5775
4,2023-01-07,AIIMS Raipur,51.333542,21.2575,81.5775,7,0.285947,0.349989,0.285947,7,...,0.639584,2.145734,1.023755,1.283327,28.675352,18.036318,47.799848,0.064576,21.2575,81.5775
